# COGS 188 - Project Proposal

# Project Description

You have the choice of doing either (1) an AI solve a problem style project or (2) run a Special Topics class on a topic of your choice.  If you want to do (2) you should fill out the _other_ proposal for that. This is the proposal description for (1).

You will design and execute a machine learning project. There are a few constraints on the nature of the allowed project. 
- The problem addressed will not be a "toy problem" or "common training students problem" like 8-Queens or a small Traveling Salesman Problem or similar
- If its the kind of problem (e.g., RL) that interacts with a simulator or live task, then the problem will have a reasonably complex action space. For instance, a wupus world kind of thing with a 9x9 grid is definitely too small.  A simulated mountain car with a less complex 2-d road and simplified dynamics seems like a fairly low achievement level.  A more complex 3-d mountain car simulation with large extent and realistic dynamics, sure sounds great!
- If its the kind of problem that uses a dataset, then the dataset will have >1k observations and >5 variables. I'd prefer more like >10k observations and >10 variables. A general rule is that if you have >100x more observations than variables, your solution will likely generalize a lot better. The goal of training an unsupervised machine learning model is to learn the underlying pattern in a dataset in order to generalize well to unseen data, so choosing a large dataset is very important.
- The project must include some elements we talked about in the course
- The project will include a model selection and/or feature selection component where you will be looking for the best setup to maximize the performance of your AI system. Generally RL tasks may require a huge amount of training, so extensive grid search is unlikely to be possible. However expoloring a few reasonable hyper-parameters may still be possible. 
- You will evaluate the performance of your AI system using more than one appropriate metric
- You will be writing a report describing and discussing these accomplishments


Feel free to delete this description section when you hand in your proposal.

# Names

- Ranya Tashkandy
- Daria Stolyarova
- Rosa Gonzalez Encinas

# Abstract 
This section should be short and clearly stated. It should be a single paragraph <200 words.  It should summarize: 
- what your goal/problem is
- what the data used represents and how they are measured
- what you will be doing with the data
- how performance/success will be measured

# Background

Fill in the background and discuss the kind of prior work that has gone on in this research area here. **Use inline citation** to specify which references support which statements.  You can do that through HTML footnotes (demonstrated here). I used to reccommend Markdown footnotes (google is your friend) because they are simpler but recently I have had some problems with them working for me whereas HTML ones always work so far. So use the method that works for you, but do use inline citations.

Here is an example of inline citation. After government genocide in the 20th century, real birds were replaced with surveillance drones designed to look just like birds<a name="lorenz"></a>[<sup>[1]</sup>](#lorenznote). Use a minimum of 3 to 5 citations, but we prefer more <a name="admonish"></a>[<sup>[2]</sup>](#admonishnote). You need enough citations to fully explain and back up important facts. 

Remeber you are trying to explain why someone would want to answer your question or why your hypothesis is in the form that you've stated. 

# Problem Statement

Clearly describe the problem that you are solving. Avoid ambiguous words. The problem described should be well defined and should have at least one ML-relevant potential solution. Additionally, describe the problem thoroughly such that it is clear that the problem is quantifiable (the problem can be expressed in mathematical or logical terms), measurable (the problem can be measured by some metric and clearly observed), and replicable (the problem can be reproduced and occurs more than once).

# Data

For our data, we will use the full dataset from the [NIJ Recidivism Forecasting Challenge](https://nij.ojp.gov/funding/recidivism-forecasting-challenge), which contains 25,835 observations and 54 columns. Each row represents a person released from prison to parole supervision in Georgia between 2013 and 2015. The full dataset includes the original training and test records with outcome labels, so we will create our own splits without combining separate files. Our target variable will be 'Recidivism_Within_3years', which records whether someone was rearrested for a new felony or misdemeanor within three years of starting supervision. We will encode its 14,904 “Yes” outcomes as 1 and its 10,931 “No” outcomes as 0. Our predictions will concern recorded rearrest, which does not establish whether someone committed another crime.

We will consider predictors such as age at release, education, prison offense, time served, and previous arrests and convictions. These variables include grouped ranges, categorical labels, binary values, and counts with upper limits. Before choosing the final inputs, we will check the codebook to confirm when each variable was collected. Since we will make predictions using information available at release, we will exclude later information such as drug-test results, employment during parole, and supervision violations. We will also remove IDs, the original 'Training_Sample' indicator, and all outcome columns from the predictors. As for race, it will be kept separately for comparing errors across groups.

During preprocessing, we will fill missing numerical values with the training-set median and treat missing categorical values as a separate category. Categorical predictors, including age and time-served ranges, will be one-hot encoded, and numerical predictors will be standardized. All preprocessing will be fitted using only the training data, then applied to the validation and test sets. We plan to use a stratified 60/20/20 training, validation, and test split with a fixed random seed. The validation set will be used to compare feature subsets during hill climbing and simulated annealing, while the test set will remain separate until the models are finalized. Repeatedly using the test set to choose features could lead us to select a model that performs well on that particular data but does not perform as well on new observations. 

# Proposed Solution

Our baseline will be logistic regression using all eligible predictors. We will compare it with logistic regression models using features selected by hill climbing and simulated annealing. We chose logistic regression because it produces rearrest probabilities and should be practical to train repeatedly during feature selection. Our goal is to determine whether either search can identify a smaller set of predictors that performs as well as or better than the baseline.

Each candidate solution will contain a subset of the original predictors, with neighboring solutions formed by adding or removing one variable. All one-hot encoded columns from a categorical predictor will stay together. Candidate models will be fitted on the training set and compared using validation Brier score, while the logistic regression settings will remain fixed across approaches.

Hill climbing will evaluate neighboring feature sets and move to the one with the greatest improvement in validation Brier score. When no improving move remains, the search will restart from a different randomly chosen subset, continuing until its budget is exhausted. Simulated annealing will randomly add or remove one predictor at each step and sometimes accept a worse score to explore beyond a local optimum. The probability of accepting a worse change will follow $e^{-\frac{\Delta}{T}}$, where $Δ$ is the increase in Brier score and $T$ is the temperature. As the temperature decreases, the search will become less likely to accept worse changes. We will retain the best feature set found during each run.
    
Before the main comparison, we will run a small pilot to choose the search budget, number of restarts, and simulated annealing temperature and cooling schedule. Both methods will receive the same maximum number of model fits per run. Previously evaluated subsets will be stored and reused without fitting the same model again, and a separate limit on search steps will prevent repeated visits from extending a run indefinitely. The final settings and stopping rules will be documented in our report.

We will run each method five times with different random seeds to check how consistently it finds useful feature sets. The data split will stay the same across runs. Implementation will use Python, pandas, and scikit-learn, with our own hill climbing and simulated annealing functions. All feature selection and adjustments to the search settings will use training and validation data. Once those choices are complete, we will compare the baseline and selected models on the untouched test set.

# Evaluation Metrics

Brier score will be our main metric for comparing the baseline with the models using selected features. We chose it because we want to evaluate rearrest probabilities, including how confident the model is when it makes an incorrect prediction. Validation Brier score will guide feature selection, and test-set Brier score will measure final performance. It is calculated as:

$$
\text{Brier score} = \frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2
$$

$N$ is the number of observations, $p_i$ is the predicted rearrest probability, and $y_i$ is the recorded outcome, either 0 or 1. So, lower Brier scores will indicate that our predicted probabilities are closer to the recorded outcomes.

We will also use balanced accuracy to evaluate the yes/no predictions at a threshold of 0.50. Since more people in the dataset were rearrested than were not, we want to give both outcome groups equal weight when scoring the models. This will add another perspective alongside Brier score, which will evaluate the predicted probabilities. Balanced accuracy gives equal weight to sensitivity and specificity. In the formula, $TP$ counts correctly predicted rearrest outcomes, while $FN$ counts rearrest outcomes incorrectly predicted as no rearrest: 

$$
\text{Balanced accuracy} = \frac{1}{2}\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right)
$$

For the group comparison, we will calculate false-positive rates separately for Black and White individuals using a fixed probability threshold of 0.50. We will report both rates, their absolute difference, and the number of people in each group. Incorrectly predicting rearrest could have consequences if a similar system were used in practice, so we want to examine whether these errors occur more often for one group. For each group, the calculation will be:

$$
\text{FPR} = \frac{FP}{FP+TN}
$$

$FP$ counts people without a recorded rearrest who receive a positive prediction, while $TN$ counts those correctly given a negative prediction. 

For each method, we will record the number of selected predictors and the search runtime on the same computing environment, then report the average results and standard deviations across the five runs. The false-positive-rate comparison between groups will cover one aspect of fairness.

# Ethics & Privacy

If your project has obvious potential concerns with ethics or data privacy discuss that here.  Almost every ML project put into production can have ethical implications if you use your imagination. Use your imagination. Get creative!

Even if you can't come up with an obvious ethical concern that should be addressed, you should know that a large number of ML projects that go into producation have unintended consequences and ethical problems once in production. How will your team address these issues?

Consider a tool to help you address the potential issues such as https://deon.drivendata.org

# Team Expectations 

Put things here that cement how you will interact/communicate as a team, how you will handle conflict and difficulty, how you will handle making decisions and setting goals/schedule, how much work you expect from each other, how you will handle deadlines, etc...
* *Team Expectation 1*
* *Team Expectation 2*
* *Team Expecation 3*
* ...

# Project Timeline Proposal

Replace this with something meaningful that is appropriate for your needs. It doesn't have to be something that fits this format.  It doesn't have to be set in stone... "no battle plan survives contact with the enemy". But you need a battle plan nonetheless, and you need to keep it updated so you understand what you are trying to accomplish, who's responsible for what, and what the expected due dates are for each item.

| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 1/20  |  1 PM |  Brainstorm topics/questions (all)  | Determine best form of communication; Discuss and decide on final project topic; discuss hypothesis; begin background research | 
| 1/26  |  10 AM |  Do background research on topic (Pelé) | Discuss ideal dataset(s) and ethics; draft project proposal | 
| 2/1  | 10 AM  | Edit, finalize, and submit proposal; Search for datasets (Beckenbaur)  | Discuss Wrangling and possible analytical approaches; Assign group members to lead each specific part   |
| 2/14  | 6 PM  | Import & Wrangle Data ,do some EDA (Maradonna) | Review/Edit wrangling/EDA; Discuss Analysis Plan   |
| 2/23  | 12 PM  | Finalize wrangling/EDA; Begin programming for project (Cruyff) | Discuss/edit project code; Complete project |
| 3/13  | 12 PM  | Complete analysis; Draft results/conclusion/discussion (Carlos)| Discuss/edit full project |
| 3/19  | Before 11:59 PM  | NA | Turn in Final Project  |

# Footnotes
<a name="lorenznote"></a>1.[^](#lorenz): Lorenz, T. (9 Dec 2021) Birds Aren’t Real, or Are They? Inside a Gen Z Conspiracy Theory. *The New York Times*. https://www.nytimes.com/2021/12/09/technology/birds-arent-real-gen-z-misinformation.html<br> 
<a name="admonishnote"></a>2.[^](#admonish): Also refs should be important to the background, not some randomly chosen vaguely related stuff. Include a web link if possible in refs as above.<br>
<a name="sotanote"></a>3.[^](#sota): Perhaps the current state of the art solution such as you see on [Papers with code](https://paperswithcode.com/sota). Or maybe not SOTA, but rather a standard textbook/Kaggle solution to this kind of problem
